# 00 · Exploración del Excel del caso

**Objetivo:** entender el contenido de `data/raw/` antes de limpiar. Este notebook **no escribe** en `data/`; solo lee, resume y guarda figuras para el informe.

> Regla: los notebooks orquestan y muestran; la lógica reutilizable vive en `src/optimum/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from optimum.reporting.figures import COLORS, new_figure, save_fig
from optimum.reporting.tables import save_table

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## Hojas del libro

In [ ]:
from optimum.io.excel import load_workbook

raw = load_workbook()
pd.DataFrame({h: df.shape for h, df in raw.items()}, index=["filas", "columnas"]).T

In [ ]:
raw["Léeme"]

In [ ]:
raw["Notes"]

## Series históricas

60 cierres de mes (2021-01 a 2025-12) de 24 factores: 6 nodos spot y 5 forwards por moneda, FX (soles por dólar) y un índice de renta variable en USD.

In [ ]:
hist = raw["Historical_Market"].set_index("Date")
hist.describe().T.round(4)

In [ ]:
fig, axes = new_figure(1, 2, aspect=0.4, sharey=True)
for ax, ccy in zip(axes, ["PEN", "USD"], strict=True):
    for n in ["ON", "1Y", "5Y", "20Y"]:
        ax.plot(hist.index, 100 * hist[f"{ccy}_SPOT_{n}"], label=n)
    ax.set_title(f"Spot {ccy}")
axes[0].set_ylabel("Tasa cero (%)")
axes[1].legend(title="Nodo")
save_fig(fig, "curvas_spot_historicas")

In [ ]:
fig, axes = new_figure(1, 2, aspect=0.4)
axes[0].plot(hist.index, hist["FX_PENUSD"], color=COLORS["USD"])
axes[0].set(title="Tipo de cambio", ylabel="PEN por USD")
axes[1].plot(hist.index, hist["EQUITY_USD"], color=COLORS["EQUITY"])
axes[1].set(title="Renta variable (USD)", ylabel="Índice")
save_fig(fig, "fx_equity_historico")

## Cambios mensuales y persistencia

La hoja `Factor_Changes` trae los cambios mensuales (Δ absoluta en tasas, log-retorno en FX y equity). Si los cambios fueran independientes, la varianza del cambio a 12 meses sería 12 veces la mensual (razón de varianzas = 1). Una razón mayor que 1 indica persistencia, y un bootstrap i.i.d. mes a mes subestimaría el riesgo anual (motiva D-09).

In [ ]:
changes = raw["Factor_Changes"].set_index("Date")
cols = ["PEN_SPOT_ON", "USD_SPOT_ON", "PEN_SPOT_10Y", "USD_SPOT_10Y", "FX_PENUSD", "EQUITY_USD"]
persistencia = pd.DataFrame(
    {
        "σ mensual": changes[cols].std(),
        "autocorr. 1 mes": changes[cols].apply(lambda s: s.autocorr(1)),
        "razón var. 12 m": changes[cols].rolling(12).sum().var() / (12 * changes[cols].var()),
    }
)
save_table(
    persistencia,
    "persistencia_factores",
    caption="Persistencia de los cambios mensuales, 2021--2025 (σ en decimales).",
    label="tab:persistencia",
    fmt="{:.4f}",
)
persistencia.round(4)

## Covarianza del Excel

Las forwards se derivan de las spot, así que la matriz de covarianza (mensual) es singular en la práctica.

In [ ]:
cov = raw["Covariance"].set_index(raw["Covariance"].columns[0]).astype(float)
w = np.linalg.eigvalsh(cov.values)[::-1]
fig, ax = new_figure(width=0.6, aspect=0.6)
ax.semilogy(np.arange(1, len(w) + 1), np.clip(w, 1e-20, None), "o-", color=COLORS["neutral"], ms=3)
ax.set(xlabel="Componente", ylabel="Autovalor (escala log)", title="Autovalores de la covarianza")
save_fig(fig, "autovalores_covarianza")
pd.Series({"λ_max": w[0], "λ_min": w[-1], "var. explicada 3 comp.": w[:3].sum() / w.sum()})

## Hallazgos

Verificados en código (limpieza, tests o notebooks 01-02) y llevados a `docs/decisiones.md`:

- [x] **Cambios de factores**: Δ absoluta en tasas y log-retorno en FX y equity; coinciden con `Factor_Changes` (error < 1e-10) → D-02.
- [x] **Covarianza singular**: los autovalores caen más de diez órdenes de magnitud porque las forwards son función de las spot. No se invierte ni se usa directamente para simular.
- [x] **Persistencia**: autocorrelación de 1 mes ≈ 0.9 en tasas y equity; razón de varianzas a 12 m de 2 a 7 (tabla anterior). El bootstrap i.i.d. mes a mes subestima el riesgo anual → bloques de 6 × 2 (D-09).
- [x] **Forwards del Excel = implícitas** de la curva spot (interpolación lineal, efectiva anual), 0.0 pb de diferencia (notebook 02) → las forwards se recalculan desde la spot shockeada (D-10).
- [x] **`Scenario_Tree_D`** tiene dos tablas apiladas (transiciones y shocks): se separan en dos parquet.
- [x] **Flujos flotantes** de `Cash_Flows`: solo son la proyección base; se reproyectan por escenario (D-13).
- [x] **`Calibration_Spreads`** es referencia: se recalibra para que V0 = monto de mercado (notebook 02).
- [x] **`notional_native`** (moneda original) ≠ **`market_value_pen`** (valor de mercado en PEN).
- [x] **Tramos de estrés** no definidos en `Stress_Scenarios_C` → D-11.